# Actividad: Estrategias de búsqueda no informada

En esta actividad implementaremos y analizaremos distintas **estrategias de búsqueda no informada** utilizando el algoritmo general de **búsqueda en árboles** presentado en *Russell y Norvig, Artificial Intelligence: A Modern Approach*.

El objetivo es observar cómo diferentes estrategias de búsqueda pueden implementarse a partir de una misma estructura general, modificando principalmente la forma en que se administra la **frontera**.

Para ello utilizaremos un entorno sencillo tipo **GridWorld**, en el cual un agente debe encontrar una secuencia de acciones que lo lleve desde un **estado inicial** hasta un **estado objetivo** dentro de una cuadrícula.

Las acciones disponibles son:

- derecha
- izquierda
- arriba
- abajo

Cada movimiento tiene un costo uniforme igual a 1.

---

## Objetivos de la actividad

Al finalizar esta actividad, el estudiante será capaz de:

- comprender la estructura del **algoritmo general de búsqueda en árboles**;
- implementar distintas estrategias de búsqueda modificando la estructura de la frontera;
- analizar el comportamiento de diferentes algoritmos de búsqueda no informada;

---

## Estrategias que se explorarán

Durante la actividad se implementarán y analizarán las siguientes estrategias:

- Breadth-First Search (**BFS**)
- Depth-First Search (**DFS**)
- Uniform Cost Search (**UCS**)

---

## Instrucciones

1. Ejecute las celdas del notebook en el orden indicado.
2. Analice cuidadosamente el código proporcionado.
3. Complete las secciones donde se solicita implementar o modificar funciones.
4. Ejecute los algoritmos y observe las soluciones encontradas.
5. Responda las preguntas de análisis al final de la actividad.

In [3]:
# Parámetros del entorno GridWorld

# Tamaño de la cuadrícula
N = 3

# Acciones disponibles para el agente
# Cada acción corresponde a un desplazamiento (dx, dy)
acciones = {
    "izquierda": (-1, 0),
    "abajo": (0, -1),
    "arriba": (0, 1),
    "derecha": (1, 0)
}

# Estado inicial del agente
estado_inicial = (1, 1)

# Estado objetivo
estado_objetivo = (N, N)

## Entorno GridWorld

En esta actividad utilizaremos un entorno sencillo tipo **GridWorld**.

El agente se mueve en una cuadrícula de tamaño $N \times N$. Cada estado se representa mediante una coordenada $(x,y)$, donde $x$ aumenta hacia la derecha y $y$ aumenta hacia arriba.

- El agente comienza en un **estado inicial**.
- El objetivo es alcanzar un **estado objetivo**.
- En cada estado, el agente puede ejecutar alguna de las siguientes acciones:
  - derecha
  - izquierda
  - arriba
  - abajo

Cada acción genera un nuevo estado siempre que el movimiento permanezca dentro de los límites de la cuadrícula.

Los estados que pueden alcanzarse desde un estado $s$ mediante una acción válida constituyen sus **sucesores**.

## Función sucesora

En un problema de búsqueda, la **función sucesora** describe las transiciones que pueden realizarse desde un estado.

Formalmente, para un estado $s$, podemos representar la función sucesora como:

$$
\text{SUCESOR-FN}(s)=\{(a,s')\}
$$

donde:

- $a$ es una acción aplicable en el estado $s$;
- $s'$ es el estado resultante de ejecutar dicha acción.

En el entorno **GridWorld**, la función sucesora analiza las cuatro acciones posibles del agente:

- derecha
- izquierda
- arriba
- abajo

Una acción se considera válida únicamente si el estado resultante permanece **dentro de los límites de la cuadrícula**.

Por lo tanto, la función sucesora devuelve únicamente los pares $(a,s')$ correspondientes a movimientos válidos.

In [4]:
# Def de funcion sucesora
def es_valido(x,y):
    return 1 <= x <= N and 1<= y <= N
    
def sucesor_fn(estado):
    x, y = estado
    sucesores = []

    for accion, (dx, dy) in acciones.items():
        nuevo_x = x + dx
        nuevo_y = y + dy

        if es_valido(nuevo_x, nuevo_y):
            sucesores.append((accion, (nuevo_x, nuevo_y)))

    return sucesores

In [5]:
# probar la función sucesora
print(sucesor_fn((1,1)))
print(sucesor_fn((2,2)))

[('arriba', (1, 2)), ('derecha', (2, 1))]
[('izquierda', (1, 2)), ('abajo', (2, 1)), ('arriba', (2, 3)), ('derecha', (3, 2))]


## Test objetivo

El **test objetivo** determina si un estado corresponde a una **solución del problema**.

Formalmente, podemos definir una función:

$$
f_{\text{objetivo}}(s)=
\begin{cases}
\text{True}, & \text{si } s \text{ satisface la condición objetivo},\\
\text{False}, & \text{en caso contrario}.
\end{cases}
$$

En este entorno **GridWorld**, el objetivo del agente es alcanzar una posición específica dentro de la cuadrícula.

Por lo tanto, el test objetivo verifica si el estado actual coincide con el **estado objetivo**.

In [6]:
# Función test objetivo
def test_objetivo(estado):
    return estado == estado_objetivo

In [7]:
#evaluacion
print(test_objetivo((3,3)))
print(test_objetivo((2,2)))

True
False


## Función de costo

En un problema de búsqueda podemos distinguir entre el **costo de una acción** y el **costo total de un camino**.

La función de costo de una acción determina el costo de realizar una transición desde un estado $s$ hacia un estado sucesor $s'$ mediante una acción $a$:

$$
c(s,a,s')
$$

En este entorno **GridWorld**, asumiremos que todos los movimientos tienen **costo uniforme**, por lo que:

$$
c(s,a,s') = 1
$$

Un camino está formado por una secuencia de transiciones:

$$
(s,a,s')
$$

y su costo total se obtiene sumando el costo de todas las acciones que lo componen:

$$
g = \sum_{(s,a,s') \in \text{camino}} c(s,a,s')
$$

Como todas las acciones tienen costo igual a 1, en este caso el costo del camino coincide con el **número de movimientos realizados**.

Esta función de costo será utilizada posteriormente por **Uniform Cost Search (UCS)** para seleccionar de la frontera el camino con menor costo acumulado.


In [8]:
# funcion Costo
def costo_accion(estado, accion, estado_sucesor):
    return 1

# función costo del camino
def costo_camino(camino):
    costo = 0
    for estado, accion, estado_sucesor in camino:
        costo += costo_accion(estado, accion, estado_sucesor)

    return costo

## Definición del problema de búsqueda

En inteligencia artificial, un **problema de búsqueda** puede describirse mediante varios componentes fundamentales:

1. **Estado inicial**: estado desde el cual comienza el agente.
2. **Función sucesora**: determina qué estados pueden alcanzarse desde un estado dado y qué acción produce cada transición.
3. **Test objetivo**: determina si un estado corresponde a una solución del problema.
4. **Función de costo**: asigna un costo a cada transición $(s,a,s')$ del entorno.

A partir de estos componentes, el algoritmo de búsqueda explora el espacio de estados con el objetivo de encontrar un **camino** desde el estado inicial hasta algún estado que satisfaga el test objetivo.

En esta implementación representaremos el problema de búsqueda mediante una estructura que agrupa estos componentes.

In [9]:
# Definición del problema
problema = {
    "estado_inicial": estado_inicial,
    "funcion_sucesora": sucesor_fn,
    "test_objetivo": test_objetivo,
    "costo_accion": costo_accion
}

## Representación de nodos en el árbol de búsqueda

Durante la ejecución de un algoritmo de búsqueda, es importante distinguir entre un **estado del problema** y un **nodo del árbol de búsqueda**.

Un estado representa una configuración posible del entorno, mientras que un nodo contiene, además del estado, información sobre el camino seguido para llegar hasta él.

En esta implementación, cada nodo almacenará:

- **estado**: estado del problema representado por el nodo;
- **padre**: nodo desde el cual fue generado;
- **acción**: acción aplicada para llegar al estado actual;
- **costo_camino**: costo acumulado desde el estado inicial hasta el nodo;
- **profundidad**: número de transiciones desde el nodo inicial.

Esta información permitirá reconstruir posteriormente la **secuencia de acciones y estados que conforma la solución** cuando se alcance un estado que satisfaga el test objetivo.

In [10]:
def hacer_nodo(estado, padre, accion, costo_camino, profundidad):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "costo_camino": costo_camino,
        "profundidad": profundidad
    }

## Expansión de nodos

Durante la búsqueda, el algoritmo explora el espacio de estados generando nuevos nodos a partir de un nodo existente.

Este proceso se conoce como **expansión de un nodo**.

Para expandir un nodo se utiliza la **función sucesora del problema**, que devuelve los pares $(a,s')$ correspondientes a las acciones válidas y a los estados resultantes.

Por cada sucesor se genera un **nuevo nodo del árbol de búsqueda** que almacena:

- el nuevo estado $s'$;
- una referencia al nodo padre;
- la acción $a$ que produjo la transición;
- el costo acumulado del camino;
- la profundidad del nuevo nodo.

Si el nodo padre tiene costo acumulado $g$ y profundidad $d$, entonces para cada sucesor:

$$
g' = g + c(s,a,s')
$$

$$
d' = d + 1
$$

De esta forma, cada nodo conserva la información necesaria para continuar la búsqueda y, posteriormente, reconstruir la solución encontrada.

In [11]:
def expandir(nodo, problema):
    sucesores = []

    for accion, estado in problema["funcion_sucesora"](nodo["estado"]):

        nuevo_nodo = hacer_nodo(
            estado,
            nodo,
            accion,
            nodo["costo_camino"] + problema["costo_accion"](
                nodo["estado"],
                accion,
                estado
            ),
            nodo["profundidad"] + 1
        )

        sucesores.append(nuevo_nodo)

    return sucesores

## Reconstrucción de la solución

Cuando el algoritmo de búsqueda encuentra un nodo cuyo estado satisface el **test objetivo**, es necesario reconstruir el camino que lleva desde el estado inicial hasta dicho nodo.

Para ello, cada nodo del árbol de búsqueda mantiene una referencia a su **nodo padre** y a la **acción** utilizada para llegar hasta él.

La reconstrucción comienza en el nodo objetivo y sigue sucesivamente las referencias al nodo padre hasta alcanzar el nodo raíz.

Durante este recorrido se registran pares de la forma:

$$
(\text{estado},\text{acción})
$$

que indican la acción ejecutada desde cada estado del camino.

Como el recorrido se realiza desde el nodo objetivo hacia el nodo inicial, al finalizar se invierte el orden de los elementos para obtener la solución desde el **estado inicial** hasta el **estado objetivo**.

In [13]:
def solucion(nodo):
    camino = []

    while nodo["padre"] is not None:
        camino.append(
            (nodo["padre"]["estado"], nodo["accion"])
        )
        nodo = nodo["padre"]

    camino.reverse()
    return camino

## Algoritmo general de búsqueda en árboles

Una vez definidos los componentes del problema y la estructura de los nodos, podemos implementar el **algoritmo general de búsqueda en árboles**.

El algoritmo mantiene una estructura llamada **frontera**, que contiene los nodos que han sido generados pero que aún no han sido expandidos.

El funcionamiento general es el siguiente:

1. Crear un nodo correspondiente al **estado inicial** e insertarlo en la frontera.
2. Mientras la frontera no esté vacía:
   - extraer un nodo de la frontera;
   - verificar si su estado satisface el **test objetivo**;
   - si satisface el objetivo, reconstruir y devolver la solución;
   - en caso contrario, **expandir el nodo** y agregar sus sucesores a la frontera.
3. Si la frontera queda vacía sin encontrar el objetivo, el problema no tiene solución alcanzable mediante la búsqueda realizada.

Un aspecto fundamental es que la **estrategia de búsqueda depende de la forma en que se seleccionan los nodos de la frontera**:

- si los nodos se extraen en orden **FIFO** (*First In, First Out*), obtenemos **Breadth-First Search (BFS)**;
- si los nodos se extraen en orden **LIFO** (*Last In, First Out*), obtenemos **Depth-First Search (DFS)**;
- si se selecciona primero el nodo con **menor costo acumulado de camino**, obtenemos **Uniform Cost Search (UCS)**.

En esta primera implementación utilizaremos **búsqueda en árboles**, por lo que un mismo estado puede aparecer en distintos nodos si se alcanza mediante caminos diferentes.

Posteriormente extenderemos el algoritmo a **búsqueda en grafos**, incorporando un mecanismo para evitar la expansión repetida de estados.

In [12]:
# Algoritmo general de búsqueda en árboles
def busqueda_arbol(problema, frontera):
    
    frontera.put([hacer_nodo(problema['estado_inicial'], None, None, 0, 0)])
    
    while frontera.pendientes():
        
        nodo = frontera.get()
        
        if problema['test_objetivo'](nodo['estado']):
            return solucion(nodo)
        
        frontera.put(expandir(nodo, problema))
        
    return None

## Frontera para Breadth-First Search (BFS)

Para implementar **Breadth-First Search (BFS)**, la frontera debe comportarse como una **cola FIFO** (*First-In, First-Out*).

Esto significa que los nodos se extraen de la frontera en el mismo orden en que fueron insertados. Como resultado, BFS expande primero los nodos de menor profundidad, recorriendo el árbol de búsqueda **nivel por nivel**.

En esta implementación utilizaremos la estructura `deque` de la biblioteca `collections`, que permite:

- insertar nuevos nodos al final de la frontera;
- extraer el siguiente nodo desde el inicio.

De esta forma, la política de selección de la frontera corresponde al comportamiento FIFO característico de BFS.

In [14]:
from collections import deque

class FronteraBFS:

    def __init__(self):
        self.fila = deque()

    def pendientes(self):
        return len(self.fila)

    def get(self):
        return self.fila.popleft()

    def put(self, nodos):
        for nodo in nodos:
            self.fila.append(nodo)

In [17]:
# Búsqueda de solución utilizando BFS
frontera_bfs = FronteraBFS()
solucion_bfs = busqueda_arbol(problema, frontera_bfs)

In [16]:
# mostrar la solución
for item in solucion_bfs:
    print(item)

((1, 1), 'arriba')
((1, 2), 'arriba')
((1, 3), 'derecha')
((2, 3), 'derecha')


## Actividad 4.1 — Implementar Depth-First Search (DFS)

En el algoritmo general de búsqueda que hemos implementado, la **estrategia de búsqueda depende de cómo se administre la frontera**.

En el caso de **Breadth-First Search (BFS)**, la frontera se comporta como una **cola FIFO** (*First-In, First-Out*), por lo que los nodos se expanden nivel por nivel.

Para implementar **Depth-First Search (DFS)**, la frontera debe comportarse como una **pila LIFO** (*Last-In, First-Out*).

Esto significa que el último nodo insertado en la frontera será el primero en ser extraído. Como resultado, DFS tiende a profundizar en una rama del árbol antes de explorar otras alternativas.

### Instrucciones

1. Implemente una nueva clase llamada `FronteraDFS`.
2. La frontera debe comportarse como una **pila LIFO**.
3. Utilice una lista de Python para almacenar los nodos pendientes.
4. Mantenga la misma interfaz utilizada en `FronteraBFS`:

   - `pendientes()`
   - `get()`
   - `put(data)`

Para implementar el comportamiento LIFO, analice qué operaciones de una lista de Python permiten:

- insertar nuevos nodos al final de la frontera;
- extraer el último nodo insertado.

Una vez implementada la frontera, ejecute el algoritmo de búsqueda utilizando:

```python
f = FronteraDFS()
sol = busqueda_arbol(problema, f)
list(sol)
```

Compare la solución obtenida con la encontrada anteriormente mediante **BFS**.

### Preguntas iniciales

1. ¿DFS encuentra una solución?
***Sí.***

2. ¿La solución tiene el mismo número de pasos que la encontrada mediante BFS?
***Sí, ambas encuentran una solución de 4 pasos.***

3. ¿La secuencia de acciones es la misma?
***No. BFS toma la ruta ((1,1), arriba) -> ((1,2), arriba) -> ((1,3), derecha) -> ((2,3), derecha), mientras que DFS toma ((1,1), derecha) -> ((2,1), derecha) -> ((3,1), arriba) -> ((3,2), arriba).***

4. ¿Qué diferencia observa entre la forma en que BFS y DFS recorren el árbol de búsqueda?
***BFS explora el árbol nivel por nivel (horiznotalmente), evualuando todas las alternativas de menor profundidad primero. DFS avanza hasta el máxima profundidad posible en la rama seleccinada (verticalmente) antes de retroceder.***

In [ ]:
# Frontera para Depth-First Search (DFS) usando comportamiento LIFO (pila)
class FronteraDFS:

    def __init__(self):
        self.pila = []

    def pendientes(self):
        return len(self.pila)

    def get(self):
        return self.pila.pop()

    def put(self, nodos):
        for nodo in nodos:
            self.pila.append(nodo)

In [19]:
# Búsqueda de solución utilizando DFS
frontera_dfs = FronteraDFS()
solucion_dfs = busqueda_arbol(problema, frontera_dfs)

In [20]:
# mostrar la solución
for item in solucion_dfs:
    print(item)

((1, 1), 'derecha')
((2, 1), 'derecha')
((3, 1), 'arriba')
((3, 2), 'arriba')


## Preguntas de análisis

Escriba una breve reflexión sobre las siguientes preguntas después de ejecutar el algoritmo con `FronteraDFS`:

1. Compare la solución obtenida con **DFS** y **BFS**.
   - ¿Se obtiene la misma secuencia de acciones?
   **Es diferente. BFS va primero hacia arriba y DFS va primero hacia la derecha.**

   - ¿Las soluciones tienen el mismo costo?
   **Sí, el costo es idéntico en ambos casos (costo=4).**

   - ¿Alguno de los algoritmos encuentra una solución con menos pasos?
   **No, ambos algoritmos encontraron la solución con el mínimo número de pasos (4 movimientos).**

2. Compare el comportamiento de ambas estrategias.
   - ¿Cómo recorre BFS el árbol de búsqueda?
   **Recorre el árbol por niveles de profundidad (en amplitud).**

   - ¿Cómo recorre DFS el árbol de búsqueda?
   **Profundiza en una sola rama del árbol hasta llegar a un estado sin sucesores o al estado objetivo (en profundidad).**

   - ¿Qué diferencia observa entre explorar por niveles y profundizar primero en una rama?
   **La diferencia es que explorar por niveles garantiza siempre la ruta con menos pasos, mientras que profundizar primero depende de totalmente del orden en que se elijan las ramas, ya que si la rama correcta está al tope de la pila, encuentra el objetivo más rápido; si no, puede perderse en ramas largas o innecesarias que incluso pueden tener una profundidad infinita.**

3. Considere el uso de memoria.
   - ¿Qué diferencia puede existir entre BFS y DFS respecto al número de nodos que permanecen almacenados en la frontera?
   **La diferencia radica en que BFS almacena todos los nodos de cada nivel en la frontera, lo que genera una complejidad exponencial de memoria O(b^d), mientras que DFS solo guarda los nodos del camino actual en la pila, con una complejidad lineal O(b*m).**

4. Considere ahora un árbol de búsqueda mucho más grande.
   - ¿Qué ventaja podría tener DFS frente a BFS?
   **Una ventaja que podría tener es que DFS ocupa menos espacio en memoria RAM que BFS.**

   - ¿Qué posible desventaja tendría profundizar demasiado antes de explorar otras alternativas?
   **La desventaja es que puede caer en ramas infinitas o muy profundas sin encontrar el objetivo y no garantiza encontrar la solución con menos pasos.**

In [22]:
# escriba su reflexión
"""
En la ejecución del entorno GridWorld, tanto BFS omo DFS encontraron una solución al estado
objetivo (3,3) con un costo total de 4 pasos. Sin embargo, DFS tomó la ruta inicial hacia arriba
mientras que DFS avanzo primero hacia la derecha (en las secuencias de acciones obtenidas 
anteriormente) debido al orden LIFO de la frontera.

Que el número de pasos fuera el mismo se atribuye al orden de expansión que se utilizó para definir 
las acciones en el problema, ya que colocar la acción "derecha" en el topde la pila facilitó el
camino para el DFS. También se observó que la diferencia fundamental radica en que BFS explora por
niveles garantizando la menor profundidad con un costo de memoria exponencial O(b^d), mientras que
DFS profundiza linealmentre O(b*m) ahorrando memoria con el riesgo de que puede quedarse una rama
muy profunda en problemas más complejos.
"""

'\nEn la ejecución del entorno GridWorld, tanto BFS omo DFS encontraron una solución al estado\nobjetivo (3,3) con un costo total de 4 pasos. Sin embargo, DFS tomó la ruta inicial hacia arriba\nmientras que DFS avanzo primero hacia la derecha (en las secuencias de acciones obtenidas \nanteriormente) debido al orden LIFO de la frontera.\n\nQue el número de pasos fuera el mismo se atribuye al orden de expansión que se utilizó para definir \nlas acciones en el problema, ya que colocar la acción "derecha" en el topde la pila facilitó el\ncamino para el DFS. También se observó que la diferencia fundamental radica en que BFS explora por\nniveles garantizando la menor profundidad con un costo de memoria exponencial O(b^d), mientras que\nDFS profundiza linealmentre O(b*m) ahorrando memoria con el riesgo de que puede quedarse una rama\nmuy profunda en problemas más complejos.\n'

## Experimento — Sensibilidad de DFS al orden de expansión

Hasta ahora, **DFS** encontró una solución utilizando el orden de acciones definido inicialmente.

Sin embargo, como la frontera de DFS se comporta como una **pila LIFO**, el orden en que se generan los sucesores puede modificar significativamente el recorrido del árbol de búsqueda.

Para observar este comportamiento, modificaremos temporalmente el orden de las acciones.

Utilice ahora:

In [ ]:
# nuevo orden de acciones para el experimento
acciones_exp = {
    "derecha": (1, 0),
    "arriba": (0, 1),
    "abajo": (0, -1),
    "izquierda": (-1, 0)
}

Antes de ejecutar nuevamente DFS, **modifique la función de búsqueda** para incorporar un parámetro que establezca el número máximo de expansiones permitidas.

La búsqueda deberá detenerse cuando alcance dicho límite, aun cuando todavía existan nodos pendientes en la frontera. Esto permitirá analizar el comportamiento de DFS sin que la ejecución continúe indefinidamente si la búsqueda entra en una rama cíclica.

In [ ]:
# nueva función sucesora que utiliza el orden de acciones modificado
def sucesor_fn_exp(estado):
    x, y = estado
    sucesores = []

    for accion, (dx, dy) in acciones_exp.items():
        nuevo_x = x + dx
        nuevo_y = y + dy

        if es_valido(nuevo_x, nuevo_y):
            sucesores.append((accion, (nuevo_x, nuevo_y)))

    return sucesores

# definición del problema para el experimento
problema_exp = {
    "estado_inicial": estado_inicial,
    "funcion_sucesora": sucesor_fn_exp,
    "test_objetivo": test_objetivo,
    "costo_accion": costo_accion
}

# algoritmo de búsqueda de árboles con límite máximo de expansiones
def busqueda_arbol_limite(problema, frontera, max_expansiones=100):
    frontera.put([hacer_nodo(problema['estado_inicial'], None, None, 0, 0)])
    expansiones = 0

    while frontera.pendientes() and expansiones < max_expansiones:
        nodo = frontera.get()

        if problema['test_objetivo'](nodo['estado']):
            return solucion(nodo)

        frontera.put(expandir(nodo, problema))
        expansiones += 1

    print("No se encontró una solución en 100 expansiones realizadas.")
    return None

In [31]:
# Ejecución de DFS con el problema modificado y límite de expansiones
frontera_dfs_exp = FronteraDFS()
solucion_dfs_exp = busqueda_arbol_limite(problema_exp, frontera_dfs_exp, max_expansiones=100)

# Si no llegó al estado objetivo pero hay nodos en la frontera, reconstruir el camino del nodo en el tope
if not solucion_dfs_exp and frontera_dfs_exp.pendientes() > 0:
    solucion_dfs_exp = solucion(frontera_dfs_exp.pila[-1])

# mostrar la solución
for item in solucion_dfs_exp:
    print(item)

((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
((1, 1), 'arriba')
((1, 2), 'abajo')
(

### Analice el resultado

1. ¿DFS continúa encontrando una solución?
**No, no encuentra una solución. El algoritmo queda atrapado en un ciclo infinito recorriendo estados repetidos y se detiene únicamente al alcanzar el límite de expansiones.**

2. ¿Qué efecto produjo modificar únicamente el orden de las acciones?
**El efecto que produjo fue que cambió el orden en que se apilan y extran los sucesores en la frontera LIFO, lo cual provocó que la búsqueda priorizara una rama con movimientos que generan un bucle entre ellos y se quedan dando vueltas antes de explorar caminos hacia el estado objetivo.**

3. Recuerde que los sucesores se insertan en una pila:
   - ¿qué sucesor será extraído primero?
   **El último sucesor insertado en la pila ("izquierda").**

   - ¿coincide con el primero que fue generado?
   **No, por la propiedad LIFO de la pila, el primer sucesor generado ("derecha") queda al fondo y se extrae al final.**

4. ¿Pueden aparecer repetidamente los mismos estados durante la búsqueda?
**Sí. Al tratarse de una búsqueda en árboles, no existe un registro de estados visitados, lo que permite volver a visitar el mismo estado múltiples veces por distintos caminos.**

5. ¿Por qué el árbol de búsqueda puede crecer indefinidamente aunque el GridWorld tenga solamente $N^2$ estados?
**Porque el árbol de búsqueda representa secuencias de acciones (caminos) y no solo estados únicos. Como se pueden revertir movimientos constantemente, la cantidad de caminos posibles se vuelve infinita.**

6. ¿El límite de expansiones elimina el problema o solamente evita que el programa se ejecute indefinidamente?
**Solamente evita que la ejecución sea infinita. Para eliminar el problema de raíz es necesario implementar búsqueda en grafos para registrar y omitir los estados ya explorados, o bien modificar el algoritmo DFS para que solo pruebe caminos hasta una determinada profundidad acotada.**

### Conclusión

Este experimento muestra que **DFS puede ser muy sensible al orden de generación de los sucesores**.

Además, en una búsqueda en árboles, un mismo estado puede generarse repetidamente mediante distintos caminos. Por esta razón, un espacio de estados finito puede producir un **árbol de búsqueda potencialmente infinito** cuando existen ciclos.

Más adelante veremos cómo la **búsqueda en grafos** permite controlar este problema registrando los estados que ya han sido explorados.

# Actividad 4.2 — Implementar Uniform Cost Search (UCS)

Hasta ahora hemos implementado distintas estrategias de búsqueda modificando la forma en que se administra la **frontera**.

- En **BFS**, la frontera se comporta como una **cola FIFO** (*First-In, First-Out*).
- En **DFS**, la frontera se comporta como una **pila LIFO** (*Last-In, First-Out*).

En esta actividad implementaremos **Uniform Cost Search (UCS)**.

En UCS, el siguiente nodo que se extrae de la frontera no se selecciona por su orden de generación ni directamente por su profundidad, sino por el **costo acumulado del camino** desde el estado inicial.

Por lo tanto, en cada iteración debe seleccionarse el nodo con menor valor de `costo_camino`.

Para implementar este comportamiento utilizaremos una **cola de prioridad** basada en un **heap**.

---

## Idea clave

Cada nodo almacena el costo acumulado desde el estado inicial en el atributo:

```python
nodo["costo_camino"]
```

Este valor se utilizará como **prioridad** dentro de la frontera.

El nodo con **menor costo acumulado** debe ser siempre el primero en ser extraído.

---

## Instrucciones

Implemente una nueva clase llamada `FronteraUCS` que utilice un **heap** para mantener los nodos organizados de acuerdo con su costo acumulado.

Puede utilizar el módulo estándar de Python `heapq`.

---

## Guía de implementación

1. Cree una estructura para almacenar el heap, por ejemplo una lista llamada `heap`.

2. Cada elemento del heap debe ser una tupla de la forma:

```python
(prioridad, contador, nodo)
```

donde:

- `prioridad` corresponde a `nodo["costo_camino"]`;
- `contador` es un número incremental utilizado para distinguir nodos que tengan la misma prioridad;
- `nodo` es el nodo del árbol de búsqueda.

El uso del contador permite conservar un criterio de desempate cuando dos o más nodos tienen el mismo costo acumulado.

3. Implemente los siguientes métodos:

### `pendientes()`

Debe devolver el número de nodos actualmente almacenados en la frontera.

### `get()`

Debe extraer del heap el nodo con **menor costo acumulado**, utilizando la operación correspondiente del módulo `heapq`.

### `put(data)`

Debe insertar en el heap todos los nodos contenidos en `data`, utilizando como prioridad el valor de `costo_camino`.

In [36]:
# FI UNAM
# Inteligencia Artificial
# Carlos Roberto Vázquez Villegas

# Clase para la frontera con UCS
import heapq

class FronteraUCS:

    def __init__(self):
        self.heap = []
        self.contador = 0

    def pendientes(self):
        return len(self.heap)

    def get(self):
        prioridad, contador, nodo = heapq.heappop(self.heap)
        return nodo


    def put(self, nodos):
        for nodo in nodos:
            prioridad = nodo["costo_camino"]
            heapq.heappush(self.heap, (prioridad, self.contador, nodo))
            self.contador += 1

## Ejecución del algoritmo

Una vez implementada la clase `FronteraUCS`, ejecute nuevamente el algoritmo general de búsqueda utilizando una instancia de esta frontera.

```python
f = FronteraUCS()
sol = busqueda_arbol(problema, f)
list(sol)
```

Observe la solución encontrada y compárela con la obtenida previamente mediante **BFS**.

Recuerde que, en este ejemplo, todas las acciones tienen costo igual a 1. Por lo tanto, el costo acumulado de un nodo coincide con su profundidad en el árbol de búsqueda.

In [38]:
# Ejecución con UCS
frontera_ucs = FronteraUCS()
solucion_ucs = busqueda_arbol(problema, frontera_ucs)

for item in solucion_ucs:
    print(item)

((1, 1), 'arriba')
((1, 2), 'arriba')
((1, 3), 'derecha')
((2, 3), 'derecha')


## Pregunta de reflexión

En este entorno, **todos los movimientos tienen costo 1**, ya que la función de costo devuelve siempre el mismo valor para cualquier transición.

1. Compare la solución obtenida mediante **UCS** con la obtenida mediante **BFS**.
2. ¿Por qué ambos algoritmos pueden producir el mismo resultado cuando todas las acciones tienen el mismo costo?
3. ¿Qué relación existe, en este caso, entre la **profundidad de un nodo** y su **costo acumulado**?
4. ¿Qué ocurriría si algunas acciones tuvieran costos diferentes?

In [39]:
# AGREGA TU REFLEXIÓN AQUÍ
"""
Refelxión sobre Uniform Cost Search (UCS):

1. Comparación co BFS:
Tanto BFS como UCS encuentran exactamente la misma solución de cuatro pasos hacia la meta (3,3).
Sin embargo, las secuencias de acciones varían, dado que BFS primero explora todo a la derecha y
luego sube a la solución, mientras que UCS explora todo hacia arriba y lueg va hacia la derecha
para encontrar la solución. Esto se debe al criterio de desempate en la cola de prioridad de UCS
cuando, como en este caso, los nodos tienen el mismo costo.

2. Razón por lo que ambos producen el mismo resultado:
Como las acciones tienen un costo uniforme de 1, el costo acumulado g(n) coincide con la
profundidad d del nodo en el árbol.

3. Relación entre profundiad y costo acumulado:
Existe una relación directamente proporcional: g(n) = d*1 = d. A mayor profundidad, mayor costo.

4. Efecto de costos diferentes:
Si las acciones de tuvieran costos diferentes, BFS buscaría únicamente el camino con menos pasos
sin considerar el costo total, mientras que UCS priorizaría los caminos de menor costo acumulado
g(n).

"""


'\nRefelxión sobre Uniform Cost Search (UCS):\n\n1. Comparación co BFS:\nTanto BFS como UCS encuentran exactamente la misma solución de cuatro pasos hacia la meta (3,3).\nSin embargo, las secuencias de acciones varían, dado que BFS primero explora todo a la derecha y\nluego sube a la solución, mientras que UCS explora todo hacia arriba y lueg va hacia la derecha\npara encontrar la solución. Esto se debe al criterio de desempate en la cola de prioridad de UCS\ncuando, como en este caso, los nodos tienen el mismo costo.\n\n2. Razón por lo que ambos producen el mismo resultado:\nComo las acciones tienen un costo uniforme de 1, el costo acumulado g(n) coincide con la\nprofundidad d del nodo en el árbol.\n\n3. Relación entre profundiad y costo acumulado:\nExiste una relación directamente proporcional: g(n) = d*1 = d. A mayor profundidad, mayor costo.\n\n4. Efecto de costos diferentes:\nSi las acciones de tuvieran costos diferentes, BFS buscaría únicamente el camino con menos pasos\nsin cons

## Reto — Costos no uniformes

Hasta ahora, todas las acciones tienen costo 1. Modifique la función de costo para que algunas acciones tengan costos diferentes.

Por ejemplo:

- moverse **arriba** cuesta 3;
- moverse **derecha** cuesta 1;
- los demás movimientos conservan costo 1.

Después de realizar la modificación:

1. Ejecute nuevamente **BFS** y **UCS** sobre el mismo problema.
2. Compare las soluciones encontradas por ambos algoritmos.
3. Calcule el costo total de cada solución.
4. Analice las siguientes preguntas:

   - ¿siguen encontrando el mismo camino?
   **No entre ellos. BFS y UCS toman caminos diferentes en este reto (arriba -> arriba -> derecha -> derecha para BFS); (derecha -> derecha -> arriba -> arriba para UCS). Sin embargo, cabe destacar que UCS sí coincide con la ruta que encontró en la actividad anterior debido a que los costos asignados favorecen avanzar primero por las acciones más baratas (derecha).**

   - ¿el camino con menos pasos es necesariamente el de menor costo?
   **No. Aunque en este ejemplo ambas soluciones tienen 4 pasos, en general, un camino corto puede incluir acciones de costo muy alto (como arriba = 3), haciendo que su costo acumulado g(n) sea mayor que el de un camino con más pasos pero formado por acciones económicas.**

   - ¿puede UCS encontrar un camino con más movimientos pero con menor costo total?
   **Sí. Como UCS expande los nodos basándose estrictamente en el menor costo acumulado g(n)$ no le importa dar más pasos siempre y cuando la suma total del costo sea menor que la de una ruta con menos pasos pero más costosa.**

   - ¿qué criterio utiliza BFS para seleccionar una solución y qué criterio utiliza UCS?
   **BFS utiliza como criterio el menor número de pasos / menor profundidad (d), mientras que UCS utiliza como criterio el menor costo acumulado total g(n). Los valores asignados a cada acción influyen totalmente en las decisiones de UCS, ya que buscan minimizar la suma de g(n).**

### Pregunta final

Explique por qué **BFS y UCS dejan de ser equivalentes cuando los costos de las acciones no son uniformes**.

**Dejan de ser equivalentes porque la profundidad d deja de ser directamente proporcional al costo acumulado g(n). BFS asume implícitamente que cada paso cuesta lo mismo (1 unitario) y busca únicamente la menor cantidad de transiciones, perdiendo la optimalidad en costo. En cambio, los criterios de costo afectan fuertemente el comportamiento de UCS, permitiéndole evaluar explícitamente g(n) en su cola de prioridad para garantizar siempre la ruta de menor costo total, sin importar cuántos pasos requiera.**

In [40]:
# Nueva función de costo con costos diferentes por acción
def costo_accion_reto(estado, accion, estado_sucesor):
    if accion == "arriba":
        return 3
    elif accion == "derecha":
        return 1
    return 1

# Definición del problema para el reto
problema_reto = {
    "estado_inicial": estado_inicial,
    "funcion_sucesora": sucesor_fn,
    "test_objetivo": test_objetivo,
    "costo_accion": costo_accion_reto
}

In [48]:
# Ejecución con BFS
frontera_bfs_reto = FronteraBFS()
solucion_bfs_reto = busqueda_arbol(problema_reto, frontera_bfs_reto)

# Costo total de la solucion
camino_bfs = []
actual_bfs = estado_inicial
for _, accion in solucion_bfs_reto:
    for x, sucesor in sucesor_fn(actual_bfs):
        if x == accion:
            camino_bfs.append((actual_bfs, accion, sucesor))
            actual_bfs = sucesor
            break

costo_total_bfs = sum(costo_accion_reto(estado, accion, sucesor) for estado, accion, sucesor in camino_bfs)

print("Solución BFS con costos no uniformes:")
for item in solucion_bfs_reto:
    print(item)

print("Costo total BFS: ", costo_total_bfs)


Solución BFS con costos no uniformes:
((1, 1), 'arriba')
((1, 2), 'arriba')
((1, 3), 'derecha')
((2, 3), 'derecha')
Costo total BFS:  8


In [49]:
# Ejecución con UCS
frontera_ucs_reto = FronteraUCS()
solucion_ucs_reto = busqueda_arbol(problema_reto, frontera_ucs_reto)

# Costo total de la solucion
camino_ucs = []
actual_ucs = estado_inicial
for _, accion in solucion_ucs_reto:
    for x, sucesor in sucesor_fn(actual_ucs):
        if x == accion:
            camino_ucs.append((actual_ucs, accion, sucesor))
            actual_ucs = sucesor
            break

costo_total_ucs = sum(costo_accion_reto(estado, accion, sucesor) for estado, accion, sucesor in camino_ucs)

print("Solución UCS con costos no uniformes:")
for item in solucion_ucs_reto:
    print(item)

print("Costo total UCS: ", costo_total_ucs)

Solución UCS con costos no uniformes:
((1, 1), 'derecha')
((2, 1), 'derecha')
((3, 1), 'arriba')
((3, 2), 'arriba')
Costo total UCS:  8
